In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive testing of fact_wf_supplier_invoice_orafin ETL logic in Databricks Unity Catalog
# Purpose: Validate PySpark conversion of SQL logic for supplier invoice fact table, including schema, data types, transformations, Delta Lake operations, and error handling
# Author: Giang Nguyen
# Date: 2025-10-27
# Description: This script tests the ETL pipeline for fact_wf_supplier_invoice_orafin, covering batch and streaming scenarios, schema validation, data type conversions, NULL handling, window functions, joins, Delta Lake upserts, and error cases. It uses test data from the provided CSV and ensures all business logic and catalog integration requirements are met.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, StringType, IntegerType, DoubleType, FloatType, ShortType, LongType, DateType, TimestampType
)
from pyspark.sql.window import Window  

# -- Setup: Define test parameters and paths
# These would typically be set via dbutils.widgets in production, but are hardcoded for test purposes
target_table_path = "/mnt/fact_wf_supplier_invoice_orafin"
partition_key = "invc_entry_dt"
table_format = "delta"
compression = "snappy"
table_name = "fact_wf_supplier_invoice_orafin"
unity_catalog = "purgo_playground"
environment = "prod"
project = "CCG"
load_type = "full"
view_unity_catalog_name = "vw_fact_wf_supplier_invoice_orafin"
raw_unity_catalog = "purgo_raw"
raw_unity_catalog_hist = "purgo_raw_hist"
config_unity_catalog = "purgo_config"
edp_lkp_unity_catalog = "purgo_edp_lkp"
dims_unity_catalog = "purgo_dims"

# -- Utility: Function to read CSV test data safely
def read_csv_to_df(file_path: str, schema: StructType) -> 'DataFrame':
    """
    Reads a CSV file into a DataFrame with the provided schema.
    Args:
        file_path (str): Path to the CSV file.
        schema (StructType): Schema for the DataFrame.
    Returns:
        DataFrame: Loaded DataFrame or empty DataFrame if file is missing/invalid.
    """
    try:
        df = spark.read.csv(file_path, header=True, schema=schema)
        return df
    except Exception as e:
        # Gracefully handle missing or invalid data
        return spark.createDataFrame([], schema)

# -- Utility: Function to assert DataFrame schema matches expected
def assert_schema(df: 'DataFrame', expected_schema: StructType) -> None:
    """
    Asserts that the DataFrame schema matches the expected schema.
    Args:
        df (DataFrame): DataFrame to check.
        expected_schema (StructType): Expected schema.
    Returns:
        None. Raises AssertionError if schema does not match.
    """
    actual_fields = [(f.name, f.dataType, f.nullable) for f in df.schema.fields]
    expected_fields = [(f.name, f.dataType, f.nullable) for f in expected_schema.fields]
    assert actual_fields == expected_fields, f"Schema mismatch: {actual_fields} != {expected_fields}"

# -- Utility: Function to validate allowed values for input parameters
def assert_allowed_value(param: str, value: str, allowed: list) -> None:
    """
    Asserts that the parameter value is in the allowed list.
    Args:
        param (str): Parameter name.
        value (str): Value to check.
        allowed (list): List of allowed values.
    Returns:
        None. Raises AssertionError if value is not allowed.
    """
    assert value in allowed, f"Value {value} not allowed for {param}"

# -- Utility: Function to validate type of input parameter
def assert_type(param: str, value, expected_type: type) -> None:
    """
    Asserts that the parameter value is of the expected type.
    Args:
        param (str): Parameter name.
        value: Value to check.
        expected_type (type): Expected Python type.
    Returns:
        None. Raises AssertionError if type does not match.
    """
    assert isinstance(value, expected_type), f"Invalid value for {param}: {value}"

# -- Utility: Function to check for duplicate upsert keys
def assert_no_duplicates(df: 'DataFrame', key_cols: list) -> None:
    """
    Asserts that there are no duplicate records for the upsert key.
    Args:
        df (DataFrame): DataFrame to check.
        key_cols (list): List of column names forming the upsert key.
    Returns:
        None. Raises AssertionError if duplicates are found.
    """
    dup_count = df.groupBy(key_cols).count().filter(F.col("count") > 1).count()
    assert dup_count == 0, f"Duplicate records found for upsert key: {key_cols}"

# -- Utility: Function to check for type mismatches in columns
def assert_column_type(df: 'DataFrame', column: str, expected_type: type) -> None:
    """
    Asserts that all non-null values in the column are of the expected type.
    Args:
        df (DataFrame): DataFrame to check.
        column (str): Column name.
        expected_type (type): Expected Python type.
    Returns:
        None. Raises AssertionError if type mismatch is found.
    """
    mismatches = df.filter(
        (F.col(column).isNotNull()) & (~F.col(column).cast(expected_type.__name__).isNotNull())
    ).count()
    assert mismatches == 0, f"Type mismatch for column {column}: expected {expected_type.__name__}"

# -- Utility: Function to validate NULL handling
def assert_null_handling(df: 'DataFrame', column: str) -> None:
    """
    Asserts that NULL values are handled correctly in the column.
    Args:
        df (DataFrame): DataFrame to check.
        column (str): Column name.
    Returns:
        None. Raises AssertionError if NULLs are not handled as expected.
    """
    # For this test, just ensure NULLs exist where expected
    null_count = df.filter(F.col(column).isNull()).count()
    assert null_count >= 0, f"NULL handling failed for column {column}"

# -- Utility: Function to validate complex types (ARRAY, STRUCT, MAP)
def assert_complex_type(df: 'DataFrame', column: str, expected_type: str) -> None:
    """
    Asserts that the column is of the expected complex type.
    Args:
        df (DataFrame): DataFrame to check.
        column (str): Column name.
        expected_type (str): Expected Spark type name ('array', 'struct', 'map').
    Returns:
        None. Raises AssertionError if type does not match.
    """
    field = df.schema[column]
    assert expected_type in str(field.dataType).lower(), f"Column {column} is not of type {expected_type}"

# -- Utility: Function to validate Delta Lake operations
def assert_delta_merge(target_path: str, source_df: 'DataFrame', merge_condition: str, update_expr: dict, insert_expr: dict) -> None:
    """
    Asserts that Delta Lake MERGE operation works as expected.
    Args:
        target_path (str): Delta table path.
        source_df (DataFrame): Source DataFrame.
        merge_condition (str): Merge condition string.
        update_expr (dict): Update expressions.
        insert_expr (dict): Insert expressions.
    Returns:
        None. Raises AssertionError if MERGE fails.
    """
    from delta.tables import DeltaTable  
    try:
        delta_table = DeltaTable.forPath(spark, target_path)
        delta_table.alias("tgt").merge(
            source=source_df.alias("src"),
            condition=merge_condition
        ).whenMatchedUpdate(set=update_expr
        ).whenNotMatchedInsert(values=insert_expr
        ).execute()
    except Exception as e:
        assert False, f"Delta MERGE failed: {str(e)}"

# -- Utility: Function to cleanup test tables
def cleanup_table(table_path: str) -> None:
    """
    Drops the Delta table at the specified path if it exists.
    Args:
        table_path (str): Path to the Delta table.
    Returns:
        None.
    """
    from delta.tables import DeltaTable  
    try:
        DeltaTable.forPath(spark, table_path).delete()
    except Exception:
        pass  # Table may not exist, ignore

# -- Define expected schema for fact_wf_supplier_invoice_orafin
expected_schema = StructType([
    StructField("document_type", StringType(), True),
    StructField("txn_ref_nbr", StringType(), True),
    StructField("invc_entry_period", StringType(), True),
    StructField("po_nbr", StringType(), True),
    StructField("po_line_nbr", StringType(), True),
    StructField("src_sys_cd", StringType(), True),
    StructField("vchr_nbr", StringType(), True),
    StructField("vchr_line_nbr", StringType(), True),
    StructField("fscl_yr_nbr", StringType(), True),
    StructField("vchr_type_cd", StringType(), True),
    StructField("vchr_status", StringType(), True),
    StructField("item_nbr", StringType(), True),
    StructField("item_desc", StringType(), True),
    StructField("thermo_item_nbr", StringType(), True),
    StructField("supplier_cd", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("supplier_type_cd", StringType(), True),
    StructField("buyer_cd", StringType(), True),
    StructField("document_desc", StringType(), True),
    StructField("invc_txn_type", StringType(), True),
    StructField("buyer_nm", StringType(), True),
    StructField("co_cd", StringType(), True),
    StructField("co_name", StringType(), True),
    StructField("hfm_entity", StringType(), True),
    StructField("business_unit", StringType(), True),
    StructField("lcr_flag", StringType(), True),
    StructField("lcr_region", StringType(), True),
    StructField("vomi_flag", StringType(), True),
    StructField("payment_compliance_flg", StringType(), True),
    StructField("po_curncy_cd", StringType(), True),
    StructField("co_curncy_cd", StringType(), True),
    StructField("post_yr_mth_nbr", StringType(), True),
    StructField("invc_entry_dt", StringType(), True),
    StructField("paymt_due_dt", StringType(), True),
    StructField("suplr_invc_dt", StringType(), True),
    StructField("aprval_dt", StringType(), True),
    StructField("txn_orig_id", StringType(), True),
    StructField("suplr_invc_nbr", StringType(), True),
    StructField("invc_apprv_id", StringType(), True),
    StructField("unit_prc", DoubleType(), True),
    StructField("invc_qty", DoubleType(), True),
    StructField("base_qty", DoubleType(), True),
    StructField("invc_txn_amt", DoubleType(), True),
    StructField("invc_co_amt", DoubleType(), True),
    StructField("invc_txn_pmar_amt", DoubleType(), True),
    StructField("invc_co_pmar_amt", DoubleType(), True),
    StructField("unit_prc_pmar_amt", DoubleType(), True),
    StructField("txn_curncy_mth_rt", DoubleType(), True),
    StructField("co_curncy_mth_rt", DoubleType(), True),
    StructField("uom_conv_factor", DoubleType(), True),
    StructField("invc_uom_cd", StringType(), True),
    StructField("base_uom_cd", StringType(), True),
    StructField("profit_cntr", StringType(), True),
    StructField("div_cd", StringType(), True),
    StructField("site_cd", StringType(), True),
    StructField("site_name", StringType(), True),
    StructField("reporting_site", StringType(), True),
    StructField("warehouse", StringType(), True),
    StructField("warehouse_nm", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("nature", StringType(), True),
    StructField("inv_flg", StringType(), True),
    StructField("inv_flg_text", StringType(), True),
    StructField("spend_type_cd", StringType(), True),
    StructField("po_paymt_terms_cd", StringType(), True),
    StructField("po_paymt_terms_desc", StringType(), True),
    StructField("suplr_paymt_terms_cd", StringType(), True),
    StructField("suplr_paymt_terms_desc", StringType(), True),
    StructField("fk_orig", StringType(), True),
    StructField("floor_stock_cd", StringType(), True),
    StructField("contract_flag", StringType(), True),
    StructField("contract_type", StringType(), True),
    StructField("contract_start_date", StringType(), True),
    StructField("contract_end_date", StringType(), True),
    StructField("erp_commondity_cd", StringType(), True),
    StructField("erp_commondity_nm", StringType(), True),
    StructField("sec_supp_cd", StringType(), True),
    StructField("part_rev_no", StringType(), True),
    StructField("cost_centre_cd", StringType(), True),
    StructField("cost_centre_nm", StringType(), True),
    StructField("vendor_mat_no", StringType(), True),
    StructField("gl_acct_id", StringType(), True),
    StructField("gl_acct_nm", StringType(), True),
    StructField("pass_through_field", StringType(), True),
    StructField("pass_through_line", StringType(), True),
    StructField("inv_line_desc", StringType(), True),
    StructField("remit_to_addr_line_1", StringType(), True),
    StructField("remit_to_addr_line_2", StringType(), True),
    StructField("remit_to_addr_line_3", StringType(), True),
    StructField("remit_to_addr_line_4", StringType(), True),
    StructField("remit_to_city_nm", StringType(), True),
    StructField("remit_to_st_cd", StringType(), True),
    StructField("remit_to_rgn_cd", StringType(), True),
    StructField("remit_to_rgn_nm", StringType(), True),
    StructField("remit_to_cntry_cd", StringType(), True),
    StructField("remit_to_cntry_nm", StringType(), True),
    StructField("suplr_nm_src", StringType(), True),
    StructField("rpt_flex1", StringType(), True),
    StructField("invc_txn_amt_clsfctn", StringType(), True),
    StructField("supplier_segment", StringType(), True),
    StructField("ap_payment_term_cd", StringType(), True),
    StructField("ap_payment_term_desc", StringType(), True),
    StructField("actual_payment_dt", StringType(), True),
    StructField("source_country", StringType(), True)
])

# -- Load test data for fact_wf_supplier_invoice_orafin
test_csv_path = "/Workspace/Repos/giangnguyendss/test9/test/GP-106_CLONE_FROM_AD_216_Converting_fact_wf_supplier_invoice_orafin_Spark_SQL_testdata.csv"
df_test = read_csv_to_df(test_csv_path, expected_schema)

# -- Test: Schema validation
assert_schema(df_test, expected_schema)

# -- Test: Data type conversions
assert_column_type(df_test, "unit_prc", float)
assert_column_type(df_test, "invc_qty", float)
assert_column_type(df_test, "invc_txn_amt", float)
assert_column_type(df_test, "invc_entry_period", str)
assert_column_type(df_test, "actual_payment_dt", str)

# -- Test: NULL handling
assert_null_handling(df_test, "document_type")
assert_null_handling(df_test, "unit_prc")
assert_null_handling(df_test, "base_qty")

# -- Test: Allowed values for table_format and compression
assert_allowed_value("table_format", table_format, ["delta", "parquet"])
assert_allowed_value("compression", compression, ["snappy", "gzip"])

# -- Test: Type validation for input parameters
assert_type("partition_key", partition_key, str)
assert_type("table_format", table_format, str)
assert_type("compression", compression, str)
assert_type("unity_catalog", unity_catalog, str)
assert_type("table_name", table_name, str)

# -- Test: Window function for latest snapshot per invoice
def test_latest_snapshot_window(df: 'DataFrame') -> None:
    """
    Tests window function for latest snapshot per invoice_id.
    Args:
        df (DataFrame): Input DataFrame with invoice_id and snapshot_captured_date.
    Returns:
        None. Asserts that only the latest snapshot per invoice_id is present.
    """
    window_spec = Window.partitionBy("vchr_nbr").orderBy(F.col("invc_entry_dt").desc())
    df_with_rownum = df.withColumn("rownum", F.row_number().over(window_spec))
    latest_df = df_with_rownum.filter(F.col("rownum") == 1)
    # Assert that for each invoice_id, only one record exists
    invoice_count = latest_df.groupBy("vchr_nbr").count().filter(F.col("count") > 1).count()
    assert invoice_count == 0, "Window function failed: Multiple latest snapshots per invoice_id"

test_latest_snapshot_window(df_test)

# -- Test: Window function for latest expense dist per distribution
def test_latest_expense_dist_window(df: 'DataFrame') -> None:
    """
    Tests window function for latest expense dist per distribution.
    Args:
        df (DataFrame): Input DataFrame with distribution keys and xla_manual_override_flag.
    Returns:
        None. Asserts that only the latest expense dist per distribution is present.
    """
    window_spec = Window.partitionBy(
        "vchr_nbr", "co_cd", "cost_centre_cd", "gl_acct_id", "invc_entry_dt", "unit_prc"
    ).orderBy(F.col("invc_txn_amt").desc())
    df_with_rownum = df.withColumn("rownum", F.row_number().over(window_spec))
    latest_df = df_with_rownum.filter(F.col("rownum") == 1)
    # Assert that for each distribution, only one record exists
    dist_count = latest_df.groupBy(
        "vchr_nbr", "co_cd", "cost_centre_cd", "gl_acct_id", "invc_entry_dt", "unit_prc"
    ).count().filter(F.col("count") > 1).count()
    assert dist_count == 0, "Window function failed: Multiple latest expense dist per distribution"

test_latest_expense_dist_window(df_test)

# -- Test: Data quality validation (no negative amounts, valid payment terms)
def test_data_quality(df: 'DataFrame') -> None:
    """
    Validates data quality rules for the fact table.
    Args:
        df (DataFrame): Input DataFrame.
    Returns:
        None. Asserts that business rules are met.
    """
    # No negative transaction amounts
    neg_count = df.filter(F.col("invc_txn_amt") < 0).count()
    assert neg_count == 0, "Negative transaction amounts found"
    # Payment terms must be in allowed set
    allowed_terms = ["Net 30", "Net 60", "Immediate", "Net 0", "Net 999"]
    invalid_terms = df.filter(~F.col("ap_payment_term_cd").isin(allowed_terms)).count()
    assert invalid_terms == 0, "Invalid payment terms found"

test_data_quality(df_test)

# -- Test: Delta Lake MERGE operation
def test_delta_merge_operation(df: 'DataFrame', target_path: str) -> None:
    """
    Tests Delta Lake MERGE operation for upsert logic.
    Args:
        df (DataFrame): Source DataFrame.
        target_path (str): Delta table path.
    Returns:
        None. Asserts that MERGE works and no duplicates exist.
    """
    from delta.tables import DeltaTable  
    # Create target table if not exists
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(target_path)
    delta_table = DeltaTable.forPath(spark, target_path)
    # Prepare source for upsert
    source_df = df.withColumn("merge_key", F.col("vchr_nbr"))
    # MERGE condition and expressions
    merge_condition = "tgt.vchr_nbr = src.merge_key"
    update_expr = {col: f"src.{col}" for col in df.columns}
    insert_expr = {col: f"src.{col}" for col in df.columns}
    # Perform MERGE
    delta_table.alias("tgt").merge(
        source=source_df.alias("src"),
        condition=merge_condition
    ).whenMatchedUpdate(set=update_expr
    ).whenNotMatchedInsert(values=insert_expr
    ).execute()
    # Assert no duplicates
    assert_no_duplicates(delta_table.toDF(), ["vchr_nbr"])

test_delta_merge_operation(df_test, target_table_path)

# -- Test: Cleanup Delta table after test
cleanup_table(target_table_path)

# -- Test: Complex type validation (simulate ARRAY, STRUCT, MAP columns)
def test_complex_types() -> None:
    """
    Tests complex type columns in DataFrame.
    Args:
        None.
    Returns:
        None. Asserts that complex types are handled.
    """
    schema = StructType([
        StructField("array_col", 
            F.ArrayType(StringType()), True),
        StructField("struct_col", 
            StructType([StructField("field1", StringType(), True), StructField("field2", IntegerType(), True)]), True),
        StructField("map_col", 
            F.MapType(StringType(), IntegerType()), True)
    ])
    data = [
        (["a", "b"], {"field1": "x", "field2": 1}, {"k1": 1, "k2": 2}),
        (None, None, None)
    ]
    df = spark.createDataFrame(data, schema)
    assert_complex_type(df, "array_col", "array")
    assert_complex_type(df, "struct_col", "struct")
    assert_complex_type(df, "map_col", "map")

test_complex_types()

# -- Test: Error handling for missing required parameter
def test_missing_param(param: str) -> None:
    """
    Tests error handling for missing required input parameter.
    Args:
        param (str): Parameter name.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        value = None
        assert value is not None, f"Missing required parameter: {param}"
    except AssertionError as e:
        assert "Missing required parameter" in str(e)

test_missing_param("target_table_path")

# -- Test: Error handling for invalid parameter value
def test_invalid_param_value(param: str, value, allowed: list) -> None:
    """
    Tests error handling for invalid parameter value.
    Args:
        param (str): Parameter name.
        value: Value to check.
        allowed (list): Allowed values.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        assert value in allowed, f"Value {value} not allowed for {param}"
    except AssertionError as e:
        assert "not allowed for" in str(e)

test_invalid_param_value("table_format", "csv", ["delta", "parquet"])

# -- Test: Error handling for missing table
def test_missing_table(table_name: str, catalog: str) -> None:
    """
    Tests error handling for missing or inaccessible table.
    Args:
        table_name (str): Table name.
        catalog (str): Catalog name.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        spark.read.table(f"{catalog}.{table_name}")
        assert False, "Table should not exist"
    except Exception as e:
        assert f"Table {catalog}.{table_name} not found" in str(e) or "not found" in str(e)

test_missing_table("nonexistent_table", "purgo_raw")

# -- Test: Error handling for missing function
def test_missing_function() -> None:
    """
    Tests error handling for missing function.
    Args:
        None.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        read_control_table  # Should be defined elsewhere
        assert False, "Function should not be available"
    except NameError as e:
        assert "read_control_table" in str(e)

test_missing_function()

# -- Test: Error handling for missing constant
def test_missing_constant() -> None:
    """
    Tests error handling for missing constant.
    Args:
        None.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        cl_control_table  # Should be defined elsewhere
        assert False, "Constant should not be defined"
    except NameError as e:
        assert "cl_control_table" in str(e)

test_missing_constant()

# -- Test: Error handling for duplicate upsert key
def test_duplicate_upsert_key(df: 'DataFrame', key_cols: list) -> None:
    """
    Tests error handling for duplicate upsert key.
    Args:
        df (DataFrame): DataFrame to check.
        key_cols (list): Upsert key columns.
    Returns:
        None. Asserts that error is raised if duplicates found.
    """
    try:
        dup_count = df.groupBy(key_cols).count().filter(F.col("count") > 1).count()
        assert dup_count == 0, f"Duplicate records found for upsert key: {key_cols}"
    except AssertionError as e:
        assert "Duplicate records found for upsert key" in str(e)

# Simulate duplicate key
df_dup = df_test.union(df_test)
test_duplicate_upsert_key(df_dup, ["vchr_nbr"])

# -- Test: Error handling for type mismatch in transformation
def test_type_mismatch(df: 'DataFrame', column: str, invalid_value, expected_type: type) -> None:
    """
    Tests error handling for type mismatch in transformation.
    Args:
        df (DataFrame): DataFrame to check.
        column (str): Column name.
        invalid_value: Value to test.
        expected_type (type): Expected type.
    Returns:
        None. Asserts that error is raised if type mismatch found.
    """
    try:
        df_invalid = df.withColumn(column, F.lit(invalid_value))
        assert_column_type(df_invalid, column, expected_type)
    except AssertionError as e:
        assert f"Type mismatch for column {column}" in str(e)

test_type_mismatch(df_test, "unit_prc", "abc", float)

# -- Test: Error handling for incomplete business logic (unfinished CTE)
def test_incomplete_cte(cte_name: str) -> None:
    """
    Tests error handling for incomplete business logic in CTE.
    Args:
        cte_name (str): Name of the CTE.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        raise Exception(f"Business logic for CTE {cte_name} is incomplete or missing")
    except Exception as e:
        assert f"Business logic for CTE {cte_name} is incomplete or missing" in str(e)

test_incomplete_cte("P05")

# -- Test: Error handling for unspecified output format/destination
def test_unspecified_output_format() -> None:
    """
    Tests error handling for unspecified output format or destination.
    Args:
        None.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        output_format = None
        assert output_format is not None, "Output format or destination not specified"
    except AssertionError as e:
        assert "Output format or destination not specified" in str(e)

test_unspecified_output_format()

# -- Test: Successful validation of allowed values for input parameters
def test_successful_allowed_value(param: str, value: str, allowed: list) -> None:
    """
    Tests successful validation of allowed values for input parameters.
    Args:
        param (str): Parameter name.
        value (str): Value to check.
        allowed (list): Allowed values.
    Returns:
        None. Asserts that value is accepted.
    """
    assert value in allowed, f"Value {value} not allowed for {param}"

test_successful_allowed_value("table_format", "delta", ["delta", "parquet"])
test_successful_allowed_value("compression", "snappy", ["snappy", "gzip"])

# -- Test: Error when input parameter value is not allowed
def test_not_allowed_value(param: str, value: str, allowed: list) -> None:
    """
    Tests error handling for not allowed input parameter value.
    Args:
        param (str): Parameter name.
        value (str): Value to check.
        allowed (list): Allowed values.
    Returns:
        None. Asserts that error is raised.
    """
    try:
        assert value in allowed, f"Value {value} not allowed for {param}"
    except AssertionError as e:
        assert f"Value {value} not allowed for {param}" in str(e)

test_not_allowed_value("table_format", "csv", ["delta", "parquet"])
test_not_allowed_value("compression", "lz4", ["snappy", "gzip"])

# -- END OF TEST SCRIPT --
# spark.stop()  # Do not stop SparkSession in Databricks